# Dates and Duplicate Records

*Parse dates using known formats and resolve repeated keys using explicit update order.*


## A Date Needs a Format and a Valid Calendar Value

`01-09-2025` is ambiguous without a declared field order. The source of that
value uses day–month–year. `datetime.strptime(text, format)` checks the calendar
and field order; `.isoformat()` writes a consistent year–month–day representation.
Replacing punctuation alone cannot settle month–day ambiguity.


In [1]:
from datetime import datetime
import pandas as pd

date_records = [
    {"name": "Jiyoung", "date_text": "2025/09/01", "date_format": "%Y/%m/%d"},
    {"name": "Hojun", "date_text": "01-09-2025", "date_format": "%d-%m-%Y"},
    {"name": "Hana", "date_text": "2025.9.1", "date_format": "%Y.%m.%d"},
]
for row in date_records:
    parsed_date = datetime.strptime(row["date_text"], row["date_format"]).date()
    print(row["name"], row["date_text"], "->", parsed_date.isoformat())

try:
    datetime.strptime("2025/02/30", "%Y/%m/%d")
except ValueError:
    print("Rejected date: 2025/02/30 is not a calendar date")


Jiyoung 2025/09/01 -> 2025-09-01
Hojun 01-09-2025 -> 2025-09-01
Hana 2025.9.1 -> 2025-09-01
Rejected date: 2025/02/30 is not a calendar date


All three valid strings describe September 1, 2025. February 30 is
rejected through a specific exception handler, so the notebook still completes.


## Duplicate Rows and Repeated Keys

Identical rows are exact duplicates. Two rows with the same student–subject
key but different scores are competing versions. These prepared rows are listed
in arrival order, represented by `update_order`; a larger value means a later
update. In a real source, use a documented timestamp or version field.


In [2]:
updates_df = pd.DataFrame(
    {
        "name": ["Jiyoung", "Hojun", "Jiyoung", "Hojun"],
        "subject": ["Mathematics", "English", "Mathematics", "English"],
        "score": [95, 80, 100, 90],
        "update_order": [1, 2, 3, 4],
    }
)
key_columns = ["name", "subject"]
print("Exact duplicate rows:", int(updates_df.duplicated().sum()))
print("Repeated keys:", int(updates_df.duplicated(key_columns).sum()))
latest_df = updates_df.sort_values("update_order").drop_duplicates(
    key_columns, keep="last"
)
print("Latest version per key:")
print(latest_df.to_string(index=False))


Exact duplicate rows: 0
Repeated keys: 2
Latest version per key:
   name     subject  score  update_order
Jiyoung Mathematics    100             3
  Hojun     English     90             4


There are no exact duplicate rows but two repeated keys. Retaining the
latest versions selects Jiyoung's 100 and Hojun's 90.


## The Same Rule with a Dictionary

A tuple key identifies a student and subject. Assigning a later record to an
existing dictionary key replaces the stored version. The input must already be
ordered correctly for this rule to mean “latest”.


In [3]:
latest_by_key = {}
for row in updates_df.sort_values("update_order").to_dict("records"):
    key = (row["name"], row["subject"])
    latest_by_key[key] = row

for key in sorted(latest_by_key):
    print(key, "->", latest_by_key[key]["score"])


('Hojun', 'English') -> 90
('Jiyoung', 'Mathematics') -> 100


Both implementations retain the same two grades. The tuple key preserves
different subjects for a student, while overwrite order chooses the version.


Declare date formats instead of guessing field order. Define a record key and a reliable update order before choosing which repeated records to retain.
